# Match Hugging Face PODCAST transcripts to official MSP-Podcast audio

This notebook uses the Hugging Face `AbstractTTS/PODCAST` dataset only as a transcript source. It keeps the official MSP-Podcast labels, partitions, worker annotations, and audio files as the ground truth.

Outputs are written under the official raw directory:

```text
data/raw/msp_podcast/Transcripts.txt
```

The transcript file uses one semicolon-delimited line per utterance:

```text
MSP-PODCAST_0001_0008.wav; transcript text
```

The parser splits only on the first semicolon, so transcripts may contain semicolons.

In [ ]:
from pathlib import Path
import io
import json
import re
import hashlib

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm

try:
    import torch
    import torchaudio
except Exception as exc:
    torch = None
    torchaudio = None
    print(f"Audio content checks will be disabled: {exc}")

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

## Configuration

Run this notebook from the project root or from the `notebooks/` folder. The expected structure is:

```text
data/raw/podcast/       # downloaded Hugging Face dataset
data/raw/msp_podcast/   # official MSP-Podcast files
```

In [ ]:
PROJECT_ROOT = Path.cwd().parent # Notebook has root from /notebooks always
assert PROJECT_ROOT.match("sailer_ser_reproduction")

HF_RAW_DIR = PROJECT_ROOT / "data" / "raw" / "podcast"
MSP_RAW_DIR = PROJECT_ROOT / "data" / "raw" / "msp_podcast"

LABELS_PATH = MSP_RAW_DIR / "Labels.txt"
if not LABELS_PATH.exists():
    LABELS_PATH = MSP_RAW_DIR / "labels.txt"

PARTITIONS_PATH = MSP_RAW_DIR / "Partitions.txt"
if not PARTITIONS_PATH.exists():
    PARTITIONS_PATH = MSP_RAW_DIR / "Partition.txt"

AUDIO_DIR = MSP_RAW_DIR / "Audio"
if not AUDIO_DIR.exists() and (MSP_RAW_DIR / "Audios").exists():
    AUDIO_DIR = MSP_RAW_DIR / "Audios"

OUT_TRANSCRIPTS = MSP_RAW_DIR / "Transcripts.txt"
REPORT_DIR = PROJECT_ROOT / "outputs" / "podcast_transcript_matching"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Audio checking reads the Hugging Face audio column and official WAV files.
# Keep the sample check on; set FULL_AUDIO_CHECK=True only if you accept a long run.
RUN_AUDIO_CHECK = True
FULL_AUDIO_CHECK = False
AUDIO_CHECK_MAX_FILES = 200
AUDIO_CHECK_RANDOM_SEED = 42

MIN_FILENAME_MATCH_RATE = 0.95

print("Project root:", PROJECT_ROOT)
print("HF raw dir:", HF_RAW_DIR)
print("MSP raw dir:", MSP_RAW_DIR)
print("Official audio dir:", AUDIO_DIR)
print("Output transcripts:", OUT_TRANSCRIPTS)

## Parse official MSP filenames and partitions

The official label file is the source of valid utterance filenames. The partition file is only used to add split metadata to the QA tables.

In [ ]:
def split_semicolon(line):
    return [part.strip() for part in line.strip().split(";") if part.strip()]


def parse_label_filenames(labels_path):
    if not labels_path.exists():
        raise FileNotFoundError(labels_path)
    pattern = re.compile(r"^(MSP-PODCAST_\d+_\d+(?:_\d+)?\.wav)\s*;")
    filenames = []
    for line in labels_path.read_text(encoding="utf-8", errors="replace").splitlines():
        match = pattern.match(line.strip())
        if match:
            filenames.append(match.group(1))
    return filenames


def parse_partitions(path):
    split_map = {}
    if not path.exists():
        return split_map
    aliases = {
        "train": "train", "training": "train",
        "dev": "validation", "development": "validation", "validation": "validation", "valid": "validation",
        "test": "test", "testing": "test",
    }
    with path.open("r", encoding="utf-8", errors="replace") as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if not line or line.startswith("#"):
                continue
            fields = split_semicolon(line)
            if len(fields) >= 2:
                split_map[fields[1]] = aliases.get(fields[0].lower(), fields[0].lower())
    return split_map


label_filenames = parse_label_filenames(LABELS_PATH)
partition_map = parse_partitions(PARTITIONS_PATH)
audio_files = {p.name for p in AUDIO_DIR.rglob("*.wav")} if AUDIO_DIR.exists() else set()

official = pd.DataFrame({"filename": label_filenames})
official["utt_id"] = official["filename"].str.replace(".wav", "", regex=False)
official["split"] = official["filename"].map(partition_map).fillna("unassigned")
official["official_audio_exists"] = official["filename"].isin(audio_files)

display(official.head())
print("Official labeled rows:", len(official))
print("Unique official filenames:", official["filename"].nunique())
print("Official audio files found:", int(official["official_audio_exists"].sum()), "/", len(official))
print("Split counts:")
print(official["split"].value_counts(dropna=False))

if official["filename"].duplicated().any():
    duplicated = official[official["filename"].duplicated(keep=False)].sort_values("filename")
    duplicated.to_csv(REPORT_DIR / "official_duplicate_filenames.csv", index=False)
    raise RuntimeError(f"Duplicate official filenames found. See {REPORT_DIR / 'official_duplicate_filenames.csv'}")

## Index Hugging Face Parquet transcript rows

This reads only filename/text columns first. It does **not** read the heavy audio column until the optional audio-content check.

In [ ]:
parquet_files = sorted(HF_RAW_DIR.rglob("*.parquet"))
if not parquet_files:
    raise FileNotFoundError(f"No parquet files found under {HF_RAW_DIR}")

print("Parquet files:", len(parquet_files))
print("First parquet:", parquet_files[0])

first_schema = pq.ParquetFile(parquet_files[0]).schema.names
print("First schema columns:")
print(first_schema)

file_candidates = ["file", "filename", "audio_file", "path", "wav", "name"]
text_candidates = ["transcription", "transcript", "text", "sentence", "normalized_text"]

def choose_column(columns, candidates, contains_terms=()):
    lower = {c.lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    for c in columns:
        cl = c.lower()
        if any(term in cl for term in contains_terms):
            return c
    return None

file_col = choose_column(first_schema, file_candidates, contains_terms=("file", "path"))
text_col = choose_column(first_schema, text_candidates, contains_terms=("trans", "text"))
audio_col = choose_column(first_schema, ["audio"], contains_terms=("audio",))

print("Detected file column:", file_col)
print("Detected text column:", text_col)
print("Detected audio column:", audio_col)

if file_col is None or text_col is None:
    raise RuntimeError("Could not detect file/text columns. Inspect the schema and set file_col/text_col manually.")

In [ ]:
def extract_filename(value):
    if value is None:
        return ""
    if isinstance(value, dict):
        for key in ("path", "filename", "file", "name"):
            if key in value and value[key] is not None:
                return Path(str(value[key])).name
        return ""
    return Path(str(value)).name


def clean_transcript(text):
    if pd.isna(text):
        return ""
    text = str(text).replace("\r", " ").replace("\n", " ").replace("\t", " ")
    return re.sub(r"\s+", " ", text).strip()


records = []
for pq_path in tqdm(parquet_files, desc="Reading HF filename/text columns"):
    table = pq.read_table(pq_path, columns=[file_col, text_col])
    df = table.to_pandas()
    df = df.rename(columns={file_col: "hf_file_raw", text_col: "transcript"})
    df["filename"] = df["hf_file_raw"].map(extract_filename)
    df["transcript"] = df["transcript"].map(clean_transcript)
    df["source_parquet"] = str(pq_path.relative_to(HF_RAW_DIR))
    df["source_parquet_abs"] = str(pq_path)
    df["row_in_file"] = np.arange(len(df))
    records.append(df[["filename", "transcript", "source_parquet", "source_parquet_abs", "row_in_file"]])

hf_index = pd.concat(records, ignore_index=True)
hf_index = hf_index[hf_index["filename"].astype(bool)].reset_index(drop=True)

print("HF rows indexed:", len(hf_index))
print("Unique HF filenames:", hf_index["filename"].nunique())
display(hf_index.head())

## Match by exact filename

This is the primary check. Official MSP filenames should match Hugging Face row filenames exactly before transcripts are trusted.

In [ ]:
hf_dups = hf_index[hf_index["filename"].duplicated(keep=False)].sort_values("filename")
if len(hf_dups):
    hf_dups.to_csv(REPORT_DIR / "hf_duplicate_filenames.csv", index=False)

conflicts = (
    hf_index.groupby("filename")["transcript"]
    .nunique(dropna=False)
    .reset_index(name="num_distinct_transcripts")
    .query("num_distinct_transcripts > 1")
)
if len(conflicts):
    conflict_rows = hf_index[hf_index["filename"].isin(conflicts["filename"])].sort_values("filename")
    conflict_rows.to_csv(REPORT_DIR / "hf_conflicting_transcripts.csv", index=False)

# For duplicates with identical transcripts, keep the first row locator.
hf_unique = (
    hf_index.sort_values(["filename", "source_parquet", "row_in_file"])
    .drop_duplicates("filename", keep="first")
    .reset_index(drop=True)
)

matched = official.merge(hf_unique, on="filename", how="left")
matched["has_hf_match"] = matched["transcript"].fillna("").astype(bool)
matched["transcript_num_chars"] = matched["transcript"].fillna("").str.len()
matched["transcript_num_words"] = matched["transcript"].fillna("").str.split().map(len)

match_rate = matched["has_hf_match"].mean()
print(f"Filename match rate: {match_rate:.2%}")
print("Matched rows:", int(matched["has_hf_match"].sum()), "/", len(matched))
print("HF duplicate filename rows:", len(hf_dups))
print("HF transcript conflicts:", len(conflicts))

display(matched.head())
display(matched["has_hf_match"].value_counts(dropna=False))

missing = matched[~matched["has_hf_match"]].copy()
missing[["filename", "split", "official_audio_exists"]].to_csv(REPORT_DIR / "missing_transcripts.csv", index=False)
matched.head(200).to_csv(REPORT_DIR / "matched_transcripts_preview.csv", index=False)

if match_rate < MIN_FILENAME_MATCH_RATE:
    raise RuntimeError(
        f"Only {match_rate:.2%} of official filenames matched HF transcripts. "
        f"Inspect {REPORT_DIR / 'missing_transcripts.csv'} before writing Transcripts.txt."
    )

## Optional audio-content check

Filename matching is necessary but not sufficient. This cell compares official MSP audio against the Hugging Face audio payload for a subset, or all matched files if `FULL_AUDIO_CHECK=True`.

The strongest check is exact compressed-byte equality when the HF row stores file bytes. If that is unavailable, the notebook decodes both signals and compares duration and waveform similarity.

In [ ]:
def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def decode_audio_bytes(data):
    if torchaudio is None:
        raise RuntimeError("torchaudio is not available")
    waveform, sample_rate = torchaudio.load(io.BytesIO(data))
    return waveform, sample_rate


def decode_hf_audio_value(value):
    exact_bytes = None
    if isinstance(value, dict):
        if value.get("bytes") is not None:
            exact_bytes = value["bytes"]
            return decode_audio_bytes(exact_bytes) + (exact_bytes,)
        if value.get("array") is not None:
            arr = np.asarray(value["array"], dtype=np.float32)
            if arr.ndim == 1:
                arr = arr[None, :]
            elif arr.shape[0] > arr.shape[-1]:
                arr = arr.T
            sr = int(value.get("sampling_rate") or value.get("sample_rate") or 16000)
            return torch.tensor(arr), sr, exact_bytes
        if value.get("path") is not None:
            p = Path(str(value["path"]))
            if p.exists():
                waveform, sr = torchaudio.load(p)
                return waveform, sr, exact_bytes
    if isinstance(value, (bytes, bytearray)):
        exact_bytes = bytes(value)
        return decode_audio_bytes(exact_bytes) + (exact_bytes,)
    raise RuntimeError(f"Unsupported HF audio value type: {type(value)}")


def read_hf_audio_cell(row):
    if audio_col is None:
        raise RuntimeError("No audio column detected in HF parquet files")
    table = pq.read_table(Path(row["source_parquet_abs"]), columns=[audio_col])
    series = table.to_pandas()[audio_col]
    return series.iloc[int(row["row_in_file"])]


def compare_one_audio(row):
    official_path = AUDIO_DIR / row["filename"]
    result = {
        "filename": row["filename"],
        "official_audio_exists": official_path.exists(),
        "source_parquet": row["source_parquet"],
        "row_in_file": int(row["row_in_file"]),
    }
    if not official_path.exists():
        result["status"] = "missing_official_audio"
        return result
    try:
        official_bytes = official_path.read_bytes()
        result["official_sha256"] = sha256_bytes(official_bytes)
        official_wav, official_sr = torchaudio.load(official_path)
        hf_value = read_hf_audio_cell(row)
        hf_wav, hf_sr, hf_bytes = decode_hf_audio_value(hf_value)

        result["official_sr"] = int(official_sr)
        result["hf_sr"] = int(hf_sr)
        result["official_frames"] = int(official_wav.shape[-1])
        result["hf_frames"] = int(hf_wav.shape[-1])
        result["official_duration_s"] = float(official_wav.shape[-1] / official_sr)
        result["hf_duration_s"] = float(hf_wav.shape[-1] / hf_sr)
        result["duration_abs_diff_s"] = abs(result["official_duration_s"] - result["hf_duration_s"])

        if hf_bytes is not None:
            result["hf_sha256"] = sha256_bytes(hf_bytes)
            result["exact_bytes_match"] = result["official_sha256"] == result["hf_sha256"]
        else:
            result["exact_bytes_match"] = False

        # Waveform comparison only when sample rates match. Avoid resampling dependency.
        if official_sr == hf_sr:
            min_frames = min(official_wav.shape[-1], hf_wav.shape[-1])
            a = official_wav.mean(dim=0)[:min_frames].float()
            b = hf_wav.mean(dim=0)[:min_frames].float()
            result["waveform_mean_abs_diff"] = float((a - b).abs().mean())
            if min_frames > 1 and float(a.std()) > 0 and float(b.std()) > 0:
                result["waveform_corr"] = float(torch.corrcoef(torch.stack([a, b]))[0, 1])
            else:
                result["waveform_corr"] = np.nan
        else:
            result["waveform_mean_abs_diff"] = np.nan
            result["waveform_corr"] = np.nan

        result["content_match_status"] = (
            bool(result.get("exact_bytes_match"))
            or (
                result["duration_abs_diff_s"] <= 0.02
                and (pd.isna(result["waveform_corr"]) or result["waveform_corr"] >= 0.999)
            )
        )
        result["status"] = "ok"
    except Exception as exc:
        result["status"] = "error"
        result["error"] = repr(exc)
    return result


if RUN_AUDIO_CHECK:
    candidates = matched[matched["has_hf_match"] & matched["official_audio_exists"]].copy()
    if FULL_AUDIO_CHECK:
        check_df = candidates
    else:
        n = min(AUDIO_CHECK_MAX_FILES, len(candidates))
        check_df = candidates.sample(n=n, random_state=AUDIO_CHECK_RANDOM_SEED) if n else candidates

    print("Audio rows to check:", len(check_df))
    audio_results = [compare_one_audio(row) for _, row in tqdm(check_df.iterrows(), total=len(check_df))]
    audio_check = pd.DataFrame(audio_results)
    audio_check.to_csv(REPORT_DIR / "audio_content_check.csv", index=False)
    display(audio_check.head())

    if len(audio_check):
        print(audio_check["status"].value_counts(dropna=False))
        if "content_match_status" in audio_check.columns:
            print(audio_check["content_match_status"].value_counts(dropna=False))
else:
    audio_check = pd.DataFrame()
    print("Audio content check skipped.")

## Write `Transcripts.txt`

Only official MSP filenames are written. Labels, worker annotations, splits, and audio paths remain owned by the official MSP files.

In [ ]:
to_write = matched[matched["has_hf_match"]].copy()
to_write = to_write[["filename", "transcript"]]

OUT_TRANSCRIPTS.parent.mkdir(parents=True, exist_ok=True)
with OUT_TRANSCRIPTS.open("w", encoding="utf-8", newline="\n") as handle:
    for _, row in to_write.iterrows():
        transcript = clean_transcript(row["transcript"])
        handle.write(f"{row['filename']}; {transcript}\n")

summary = {
    "official_labeled_rows": int(len(official)),
    "official_unique_filenames": int(official["filename"].nunique()),
    "official_audio_exists": int(official["official_audio_exists"].sum()),
    "hf_rows_indexed": int(len(hf_index)),
    "hf_unique_filenames": int(hf_index["filename"].nunique()),
    "matched_transcripts": int(matched["has_hf_match"].sum()),
    "missing_transcripts": int((~matched["has_hf_match"]).sum()),
    "match_rate": float(match_rate),
    "hf_duplicate_filename_rows": int(len(hf_dups)),
    "hf_conflicting_transcript_filenames": int(len(conflicts)),
    "transcripts_path": str(OUT_TRANSCRIPTS),
    "report_dir": str(REPORT_DIR),
}
if RUN_AUDIO_CHECK and len(audio_check):
    summary["audio_check_rows"] = int(len(audio_check))
    summary["audio_check_status_counts"] = audio_check["status"].value_counts(dropna=False).to_dict()
    if "content_match_status" in audio_check.columns:
        summary["audio_content_match_counts"] = audio_check["content_match_status"].value_counts(dropna=False).to_dict()

(REPORT_DIR / "transcript_match_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

print(f"Wrote {len(to_write)} transcript lines to {OUT_TRANSCRIPTS}")
print(f"Wrote report to {REPORT_DIR / 'transcript_match_summary.json'}")
summary

## Next step

After `Transcripts.txt` is created, regenerate the official manifests with:

```bash
python scripts/create_msp_manifests.py
```

The updated script will automatically pick up `data/raw/msp_podcast/Transcripts.txt` and add transcript columns to all manifest CSVs.